# Lesson 2: Frequency and Resampling in Pandas

## 1. Learning Objectives

In this lesson, we explore how pandas handles the frequency of time-series observations and how to convert data from one time frequency to another.

By the end of this lesson, I should be able to:

- Understand the difference between regular and irregular time-series data.
- Identify the frequency of a time series using `pd.infer_freq()`.
- Understand how pandas frequency aliases work.
- Use `resample()` to aggregate time-series data into different time intervals.
- Apply aggregation methods such as `sum()`, `mean()`, `min()`, and `max()`.
- Use `agg()` to apply different aggregation methods to different columns.
- Understand the difference between `resample()` and `asfreq()`.
- Explain upsampling and downsampling.
- Understand how missing timestamps and incomplete time periods affect resampling results.

## 2. What Is Frequency in a Time Series?

Frequency describes how often observations are recorded.

Examples include:

- Hourly temperature measurements.
- Daily production quantities.
- Weekly sales figures.
- Monthly revenue.
- Annual energy consumption.

A time series is regular when its observations follow a consistent time interval. An irregular time series contains observations recorded at inconsistent intervals.

Pandas provides `pd.infer_freq()` to identify a likely frequency from a `DatetimeIndex`. It does not modify the data, and it may return `None` when the frequency cannot be inferred.

## 3. Frequency Aliases

Pandas uses frequency aliases to represent time intervals. Examples include:

| Alias | Meaning |
|---|---|
| `D` | Daily |
| `h` | Hourly |
| `min` | Every minute |
| `W` | Weekly, ending Sunday by default |
| `ME` | Month-end |
| `MS` | Month-start |
| `QE` | Quarter-end |
| `YE` | Year-end |

Modern pandas versions use aliases such as `ME`, `QE`, `YE`, and lowercase `h`. Some older aliases used in tutorials may produce deprecation warnings.

## 4. Resampling Time-Series Data

Resampling changes the time frequency by grouping observations into time-based intervals and applying an aggregation method.

For example:

- Daily production can be aggregated into weekly or monthly production totals.
- Daily temperatures can be aggregated into monthly average temperatures.
- Hourly measurements can be summarized into daily statistics.

The choice of aggregation method depends on what the data represents. Summing interval production quantities can be appropriate, while summing temperatures generally is not.

## 5. Resampling vs. `asfreq()`

These operations serve different purposes:

- `resample()` groups observations into time bins and allows aggregation.
- `asfreq()` aligns the series to a specified frequency without aggregating the observations.

When new timestamps are introduced, their values may be missing. Methods such as forward filling can fill those gaps, but this involves an assumption about how values behave between observations.

## 6. Upsampling and Downsampling

**Downsampling** converts a higher-frequency time series into a lower-frequency series, such as daily production into monthly totals.

**Upsampling** converts a lower-frequency time series into a higher-frequency index, such as observations recorded every three days into a daily index.

Upsampling does not automatically estimate missing values. Filling or interpolation must be considered separately.

## 7. Practical Context

Frequency and resampling are important in time-series analysis because real-world datasets often contain observations at different time intervals.

In chemical engineering, examples include:

- Aggregating daily plant production into monthly production totals.
- Calculating average temperatures from frequent sensor measurements.
- Converting process measurements to a consistent time interval before analysis.
- Preparing time-series data for forecasting and machine-learning models.

The correct frequency and aggregation method must always reflect the meaning of the underlying measurements.

In [1]:
import pandas as pd
import numpy as np

dates = pd.date_range(
    start="2026-01-01",
    periods=10,
    freq="D"
)

data = {
    "date": dates,
    "temperature": [25, 26, 24, 27, 28, 26, 29, 30, 28, 31],
    "production": [100, 110, 95, 120, 125, 115, 130, 140, 135, 150]
}

df = pd.DataFrame(data)
df = df.set_index("date")

df

,temperature,production
date,,
2026-01-01,25,100
2026-01-02,26,110
2026-01-03,24,95
2026-01-04,27,120
2026-01-05,28,125
2026-01-06,26,115
2026-01-07,29,130
2026-01-08,30,140
2026-01-09,28,135


In [2]:
# Understand the index
df.index


DatetimeIndex(['2026-01-01', '2026-01-02', '2026-01-03', '2026-01-04',
               '2026-01-05', '2026-01-06', '2026-01-07', '2026-01-08',
               '2026-01-09', '2026-01-10'],
              dtype='datetime64[us]', name='date', freq=None)

In [3]:
# Find the frequency of the index
pd.infer_freq(df.index)

'D'

pd.infer_freq() tries to identify the regular pattern in a sequence of dates. It may return None if there are too few timestamps or the intervals do not reveal a consistent pattern.
Important: inferring a frequency does not automatically change your dataset. It only attempts to identify the pattern.

In [4]:
# Downsampling with resample
monthly_production = df["production"].resample("ME").sum()

monthly_production

date
2026-01-31    1220
Freq: ME, Name: production, dtype: int64

- .resample("ME") groups observations into month-end-anchored monthly periods.
- .sum() adds the production values within each group.
Since all 10 observations belong to January 2026, the result will contain one row for January, with a production total of 1,220.

The key idea is that resample() groups observations into time-based bins. You then choose how to summarize the values in each bin.

In [5]:
mean_monthly_temp = df["temperature"].resample("ME").mean()
mean_monthly_temp

date
2026-01-31    27.4
Freq: ME, Name: temperature, dtype: float64

In [6]:
# It can work with other aggregate functions
df["production"].resample("ME").mean()
df["production"].resample("ME").max()
df["production"].resample("ME").min()

date
2026-01-31    95
Freq: ME, Name: production, dtype: int64

In [7]:
# Resampling multiple columns
# We can apply different aggregation to different columns

monthly_summary = df.resample("ME").agg(
    {
        "temperature": ["mean", "max", "min"],
        "production": "sum"
    }
)
monthly_summary

temperature         production
                  mean max min        sum
date                                     
2026-01-31        27.4  31  24       1220

When resampling nultiple columns use .agg and a dictionary containing all the necessary aggregate function

One important distinction: resampling does not automatically mean taking an average. It creates time-based groups, and the aggregation method determines what happens to the values.
Mini-challenge: Think like a data analyst
Using the same df DataFrame, write code for these three tasks:
1. Calculate the maximum daily temperature recorded in each week.
2. Calculate the average daily production for each week.
3. Calculate the monthly minimum temperature and monthly total production together using .agg().

In [8]:
# For question 1
df['temperature'].resample('W').max()
# For question 2
df["production"].resample("W").mean()
# For quation 3
df.resample("ME").agg(
    {
        "temperature": "min",
        "production": "sum"
    }
)

,temperature,production
date,,
2026-01-31,24,1220


Pandas uses weeks ending on Sunday by default. That means the first and last weekly groups may contain fewer than seven observations.
Resampling groups observations by calendar boundaries, not simply into consecutive chunks of seven rows. That's an important distinction when analyzing industrial data.

# resample() vs asfreq()
- resample("D").mean() creates daily bins and calculates their mean. Empty daily bins have no observations, so their result is NaN.
- asfreq("D") reindexes the data to daily frequency. Dates that weren't present receive NaN.

For this simple example, the missing dates look similar in both results, but the operations have different purposes.

- resample():	Group observations into time intervals and aggregate them
- asfreq():	Change the index frequency without aggregating observations

In [9]:
# Start with another dataset

import pandas as pd

dates = ["2026-01-01", "2026-01-03", "2026-01-05"]
temperature = [25, 27, 29]

df_demo = pd.DataFrame({
    "date": pd.to_datetime(dates),
    "temperature": temperature
}).set_index("date")

df_demo

,temperature
date,
2026-01-01,25
2026-01-03,27
2026-01-05,29


In [10]:
# Using asfreq
df_demo.asfreq("D")

,temperature
date,
2026-01-01,25.0
2026-01-02,NaN
2026-01-03,27.0
2026-01-04,NaN
2026-01-05,29.0


Pandas creates a daily index. January 2 and 4 appear with NaN because no measurements exist for those dates.
asfreq() does not calculate new temperatures. It aligns the existing observations to the requested frequency.

In [11]:
# Using resample
df_demo.resample("D").mean()

,temperature
date,
2026-01-01,25.0
2026-01-02,NaN
2026-01-03,27.0
2026-01-04,NaN
2026-01-05,29.0


This also produces a daily index with missing values on January 2 and 4 in this example. However, resample() creates time bins and aggregates observations within each bin. Here, each bin contains at most one observation, so the mean equals the original value wherever data exists.

In [12]:
# Using ffill to fill in NaN values
df_demo.asfreq("D").ffill()

,temperature
date,
2026-01-01,25.0
2026-01-02,25.0
2026-01-03,27.0
2026-01-04,27.0
2026-01-05,29.0


This assumes the last recorded value remains applicable until a new measurement arrives. That can be reasonable for some measurements, but it isn't automatically correct for every dataset.

# Lesson 2: Summary and Knowledge Check

## 1. Key Takeaways

### Frequency

Frequency describes the interval between observations in a time series. It can be regular or irregular.

```python
pd.infer_freq(df.index)
```

This attempts to infer the likely frequency from a `DatetimeIndex`. It does not change the data.

### Resampling

The `resample()` method groups observations into time-based bins and applies an aggregation function.

```python
# Monthly production total
df["production"].resample("ME").sum()

# Monthly average temperature
df["temperature"].resample("ME").mean()

# Weekly maximum temperature
df["temperature"].resample("W").max()
```

The aggregation method must match the meaning of the data.

### Multiple Aggregations

The `agg()` method allows different statistics to be calculated for different columns.

```python
df.resample("ME").agg({
    "temperature": ["mean", "min", "max"],
    "production": "sum"
})
```

### `resample()` vs. `asfreq()`

| Method | Purpose |
|---|---|
| `resample()` | Groups observations into time bins and supports aggregation |
| `asfreq()` | Aligns data to a specified frequency without aggregation |

For example:

```python
df_demo.asfreq("D")
df_demo.resample("D").mean()
```

Both may show missing values for dates without observations, but their purposes differ.

### Handling Missing Values

Forward filling uses the previous valid observation to fill a missing value.

```python
df_demo.asfreq("D").ffill()
```

Forward filling assumes that carrying the previous value forward is reasonable. This may be appropriate in some cases but not others.

### Upsampling and Downsampling

- **Downsampling:** Converting higher-frequency data to lower-frequency data, such as daily to monthly.
- **Upsampling:** Converting lower-frequency data to a higher-frequency index, such as three-day observations to a daily index.

Upsampling can introduce missing values. It does not automatically interpolate or fill them.

### Incomplete Time Periods

With `resample("W")`, pandas uses weekly bins ending on Sunday by default.

If a dataset starts on Wednesday, the first weekly bin may contain observations only from Wednesday through Sunday. The resulting total represents the observations available in that partial period, not necessarily a complete seven-day total.

### Important Practical Consideration

Before aggregating data, identify what each observation represents.

- Interval production quantities may be summed.
- Temperatures are often summarized using averages, minima, or maxima.
- Cumulative production counters generally require calculating differences to obtain production over individual intervals.

Incorrect aggregation can produce misleading results even when the code runs successfully.

---

## 2. Knowledge Check

Answer these questions without looking at the lesson notes.

### Conceptual Questions

1. What is the difference between regular and irregular time-series data?

2. What does `pd.infer_freq()` do? Does it modify the original dataset?

3. What is the difference between `resample()` and `asfreq()`?

4. Explain the difference between upsampling and downsampling. Give one example of each.

5. Why might `resample("W").sum()` produce a first weekly total that represents fewer than seven days of observations?

6. Why is summing temperature values generally inappropriate, while calculating an average temperature is often useful?

7. What assumption does forward filling make when it fills a missing value?

### Coding Questions

**8.** Write code to calculate monthly total production.

**9.** Write code to calculate the weekly maximum temperature.

**10.** Write code to calculate monthly minimum temperature and monthly total production in the same result.

**11.** Given a DataFrame `df_demo` with a `DatetimeIndex` and missing dates, write code to create a daily frequency and forward-fill missing values.

**12.** Write code to calculate hourly total production, assuming each observation represents production during its interval.

---

## 3. Self-Assessment

After completing the exercises, assess whether I can:

- [ ] Identify common pandas frequency aliases.
- [ ] Infer the likely frequency of a time series.
- [ ] Resample data using suitable aggregation methods.
- [ ] Apply multiple aggregation functions using `agg()`.
- [ ] Explain the difference between `resample()` and `asfreq()`.
- [ ] Distinguish upsampling from downsampling.
- [ ] Explain the consequences of missing timestamps and incomplete periods.
- [ ] Select an aggregation method based on the physical meaning of the data.

## 4. Final Reflection

The main lesson is that resampling is not merely a pandas syntax operation. It is a decision about how observations should be grouped and interpreted over time.

Before resampling any dataset, I should ask:

1. What does each observation represent?
2. What time interval does it cover?
3. Should the values be summed, averaged, or summarized another way?
4. Will resampling introduce missing values or partial periods?
5. Are any filling or interpolation assumptions scientifically reasonable?

These questions are especially important when working with industrial process data, sensor measurements, and chemical engineering applications.